# The Price is Right

## Week 8 Order of Play

Day 1: Modal.com and SpecialistAgent  
Day 2: RAG, FrontierAgent, Ensemble Agent  
Day 3: ScannerAgent, MessengerAgent  
Day 4: AutonomousPlannerAgent and DealAgentFramework  
Day 5: The Price Is Right Finale

## RAG (Retrieval Augmented Generation) based on a dataset of 800,000 scraped Amazon products

#### For our 2nd agent, we will be asking OpenAI to estimate the price of one of our deals - and we will give it a hand.

We discovered that LLMs are really good at this, out of the box.

And we discovered that we can beat a frontier LLM by fine-tuning an open-source LLM.

Now we are going to try **inference time** techniques instead of training -- by using RAG!

In [1]:
# imports

import os
import logging
from dotenv import load_dotenv
from huggingface_hub import login
import numpy as np
import re
from sentence_transformers import SentenceTransformer
import chromadb
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from litellm import completion
from tqdm.notebook import tqdm
from agents.evaluator import evaluate
from agents.items import Item

In [2]:
# environment

load_dotenv(override=True)
DB = "products_vectorstore_bucket"

In [3]:
# Log in to HuggingFace
# If you don't have a HuggingFace account, you can set one up for free at www.huggingface.co
# And then add the HF_TOKEN to your .env file as explained in the project README

hf_token = os.environ["HF_TOKEN"]
login(token=hf_token, add_to_git_credential=False)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
LITE_MODE = False

In [5]:
username = "Hatshe"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(
    f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items"
)

Loaded 800,000 training items, 10,000 validation items, 10,000 test items


# Now create a Chroma Datastore

Now we will use the free, open-source Vector database Chroma.  
We will create a Chroma datastore with 400,000 products from our training dataset.

In [6]:
client = chromadb.PersistentClient(path=DB)

# Introducing the SentenceTransformer Encoding LLM

The all-MiniLM is a very useful model from HuggingFace that maps sentences & paragraphs to 384 dimensional vectors and is ideal for tasks like semantic search.

https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

It can run pretty quickly locally.

As an alternative, OpenAI provides a closed-source Embeddings model. Benefits compared to OpenAI embeddings:
1. It's free and fast!
3. We can run it locally, so the data never leaves our box - might be useful if you're building a personal RAG

In [7]:
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [8]:
def price_bucket(price):
    if price < 30:
        return "0-30"
    elif price < 60:
        return "30-60"
    elif price < 100:
        return "60-100"
    elif price < 200:
        return "100-200"
    elif price < 400:
        return "200-400"
    else:
        return "400-1000"

## With that background, let's populate our Chroma database

### By calculating vectors for 800,000 scraped products

This takes 30 minutes on my machine on my GPU - it might take longer for you - feel free to use the Lite dataset!

In [9]:
# Check if the collection exists; if not, create it

collection_name = "products_with_price_bucket"
existing_collection_names = [
    collection.name for collection in client.list_collections()
]

if collection_name not in existing_collection_names:
    collection = client.create_collection(collection_name)
    for i in tqdm(range(0, len(train), 1000)):
        documents = [
            f"{item.summary}\nPrice bucket: {price_bucket(item.price)}"
            for item in train[i : i + 1000]
        ]
        vectors = encoder.encode(documents).astype(float).tolist()
        metadatas = [
            {
                "category": item.category,
                "price": item.price,
                "price_bucket": price_bucket(item.price),
            }
            for item in train[i : i + 1000]
        ]
        ids = [f"doc_{j}" for j in range(i, i + 1000)]
        ids = ids[: len(documents)]
        collection.add(
            ids=ids, documents=documents, embeddings=vectors, metadatas=metadatas
        )

collection = client.get_or_create_collection(collection_name)

# Let's visualize the vectorized data

In [ ]:
# It is very fun turning this up to 800_000 and seeing the full dataset visualized,
# but it almost crashes my box every time so do that at your own risk!! 10_000 is safe!

MAXIMUM_DATAPOINTS = 10_000

In [ ]:
CATEGORIES = [
    "Appliances",
    "Automotive",
    "Cell_Phones_and_Accessories",
    "Electronics",
    "Musical_Instruments",
    "Office_Products",
    "Tools_and_Home_Improvement",
    "Toys_and_Games",
]


COLORS = ["cyan", "blue", "brown", "orange", "yellow", "green", "purple", "red"]

In [ ]:
# Prework
result = collection.get(
    include=["embeddings", "documents", "metadatas"], limit=MAXIMUM_DATAPOINTS
)
vectors = np.array(result["embeddings"])
documents = result["documents"]
categories = [metadata["category"] for metadata in result["metadatas"]]
colors = [COLORS[CATEGORIES.index(c)] for c in categories]

In [ ]:
# Let's try a 2D chart
# TSNE stands for t-distributed Stochastic Neighbor Embedding - it's a common technique for reducing dimensionality of data

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

In [ ]:
# Create the 2D scatter plot
fig = go.Figure(
    data=[
        go.Scatter(
            x=reduced_vectors[:, 0],
            y=reduced_vectors[:, 1],
            mode="markers",
            marker=dict(size=4, color=colors, opacity=0.7),
            text=[
                f"Category: {c}<br>Text: {d[:50]}..."
                for c, d in zip(categories, documents)
            ],
            hoverinfo="text",
        )
    ]
)

fig.update_layout(
    title="2D Chroma Vectorstore Visualization",
    scene=dict(xaxis_title="x", yaxis_title="y"),
    width=1200,
    height=800,
    margin=dict(r=20, b=10, l=10, t=40),
)

fig.show()

In [ ]:
# Let's try 3D!

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

In [ ]:
# Create the 3D scatter plot
fig = go.Figure(
    data=[
        go.Scatter3d(
            x=reduced_vectors[:, 0],
            y=reduced_vectors[:, 1],
            z=reduced_vectors[:, 2],
            mode="markers",
            marker=dict(size=2, color=colors, opacity=0.7),
            text=[
                f"Category: {c}<br>Text: {d[:50]}..."
                for c, d in zip(categories, documents)
            ],
            hoverinfo="text",
        )
    ]
)

fig.update_layout(
    title="3D Chroma Vector Store Visualization",
    scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z"),
    width=1200,
    height=800,
    margin=dict(r=20, b=10, l=10, t=40),
)

fig.show()

In [10]:
def vector(item):
    return encoder.encode(item.summary)

In [11]:
def find_similars(item):
    vec = vector(item)
    results = collection.query(query_embeddings=vec.astype(float).tolist(), n_results=5)
    documents = results["documents"][0][:]
    prices = [m["price"] for m in results["metadatas"][0][:]]
    return documents, prices

In [12]:
# We need to give some context to GPT-5.1 by selecting 5 products with similar descriptions


def make_context(similars, prices):
    message = "For context, here are some other items that might be similar to the item you need to estimate.\n\n"
    for similar, price in zip(similars, prices):
        message += f"Potentially related product:\n{similar}\nPrice is ${price:.2f}\n\n"
    return message

In [13]:
documents, prices = find_similars(test[0])
print(make_context(documents, prices))

For context, here are some other items that might be similar to the item you need to estimate.

Potentially related product:
Title: Old Blood Noise Endeavors Procession Reverb  
Category: Audio Effects  
Brand: Old Blood Noise Endeavors  
Description: A compact, sci‑fi inspired reverb pedal with three modulation modes for creating otherworldly echo effects.  
Details: Features adjustable mix, decay, speed, depth, and footswitches for bypass and hold; powered by 9 V DC with 60 mA draw.
Price bucket: 200-400
Price is $209.00

Potentially related product:
Title: Old Blood Noise Endeavors Mondegreen Delay Pedal  
Category: Musical Instruments / Effects Pedals  
Brand: Old Blood Noise  
Description: A digital delay pedal that transforms your signal into creative, modulated echoes.  
Details: Features a three-way toggle for Stutter, Whirl, and Sheer modes, time/feedback/mix/morph controls, expression pedal input, 9 V DC power, and a lightweight white design.
Price bucket: 200-400
Price is $2

In [14]:
def messages_for(item, similars, prices):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}\n\n"
    message += make_context(similars, prices)
    return [{"role": "user", "content": message}]

In [15]:
documents, prices = find_similars(test[0])
print(messages_for(test[0], documents, prices)[0]["content"])

Estimate the price of this product. Respond with the price, no explanation

Title: Excess V2 Distortion/Modulation Pedal  
Category: Music Pedals  
Brand: Old Blood Noise  
Description: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  
Details: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.

For context, here are some other items that might be similar to the item you need to estimate.

Potentially related product:
Title: Old Blood Noise Endeavors Procession Reverb  
Category: Audio Effects  
Brand: Old Blood Noise Endeavors  
Description: A compact, sci‑fi inspired reverb pedal with three modulation modes for creating otherworldly echo effects.  
Details: Features adjustable mix, decay, speed, depth, and footswitches for bypass and hold; powered by 9 V 

In [16]:
# The function for gpt-5-mini


def gpt_5__1_rag(item):
    documents, prices = find_similars(item)
    response = completion(
        model="gpt-5.1",
        messages=messages_for(item, documents, prices),
        reasoning_effort="none",
        seed=42,
    )
    return response.choices[0].message.content

In [17]:
import modal


Pricer = modal.Cls.from_name("pricer-service", "Pricer")


pricer = Pricer()

In [18]:
def specialist(item):
    return pricer.price.remote(item.summary)

In [19]:
def get_price(reply):
    reply = reply.replace("$", "").replace(",", "")
    match = re.search(r"[-+]?\d*\.\d+|\d+", reply)
    return float(match.group()) if match else 0

## Download the Neural Network weights from Week 6 into this directory

The file `deep_neural_network.pth` here:

https://drive.google.com/drive/folders/1uq5C9edPIZ1973dArZiEO-VE13F7m8MK?usp=drive_link

In [20]:
from agents.deep_neural_network import DeepNeuralNetworkInference

runner = DeepNeuralNetworkInference()
runner.setup()
runner.load("deep_neural_network.pth")


def deep_neural_network(item):
    return runner.inference(item.summary)

In [33]:
# Update these three numbers after grid search on validation set
RAG_WEIGHT = 0.70
SPECIALIST_WEIGHT = 0.25
DNN_WEIGHT = 0.05

assert abs((RAG_WEIGHT + SPECIALIST_WEIGHT + DNN_WEIGHT) - 1.0) < 1e-9


def ensemble(item):
    price1 = get_price(gpt_5__1_rag(item))
    price2 = specialist(item)
    price3 = deep_neural_network(item)
    return price1 * RAG_WEIGHT + price2 * SPECIALIST_WEIGHT + price3 * DNN_WEIGHT

In [21]:
# caching pass on val set
from log_utils import cache_component_predictions

cache_component_predictions(
    items=val,
    rag_fn=lambda item: get_price(gpt_5__1_rag(item)),
    specialist_fn=specialist,
    dnn_fn=deep_neural_network,
    cache_path="logs/val_component_preds.csv",
    size=500,  # or full val later
)

Cached 25/500 items...
Cached 50/500 items...
Cached 75/500 items...
Cached 100/500 items...
Cached 125/500 items...
Cached 150/500 items...
Cached 175/500 items...
Cached 200/500 items...
Cached 225/500 items...
Cached 250/500 items...
Cached 275/500 items...
Cached 300/500 items...
Cached 325/500 items...
Cached 350/500 items...
Cached 375/500 items...
Cached 400/500 items...
Cached 425/500 items...
Cached 450/500 items...
Cached 475/500 items...
Cached 500/500 items...
Wrote 500 rows -> logs\val_component_preds.csv


,idx,title,actual,p1_rag,p2_specialist,p3_dnn
0,0,"SAFUEL Magnetic Portable Charger, 10000mAh 20W...",32.99,29.99,30.0,38.302410
1,1,Platinum Fountain Pen PROCYON Porcelain White ...,38.00,39.99,33.0,37.135178
2,2,"STARMOON Motorcycle Helmet for Men, DOT Certif...",144.99,124.99,110.0,156.536789
3,3,EWH Portable Foot Rest,29.99,19.99,19.0,44.064789
4,4,"Inwalltech M525.1LCR - 5 1/4"" 125 Watts Home T...",139.00,189.00,169.0,183.211655
...,...,...,...,...,...,...
495,495,Genuine Acura Accessories 08P42-TX4-200 Cargo ...,163.83,129.99,130.0,240.416031
496,496,"Champion Cooling, 4 Row All Aluminum Radiator ...",349.00,319.00,314.0,256.332550
497,497,"KUNFINE Tesla Style 10.4"" Android Radio CarPla...",266.00,189.99,252.0,353.458466
498,498,"Elkay ECTRU17179TC Sink, 19"", 0",218.71,259.00,252.0,301.710846


In [28]:
# offline weights search
from log_utils import grid_search_ensemble_weights

best, top = grid_search_ensemble_weights(
    "logs/val_component_preds.csv", step=0.05, top_k=10
)
top
best

Best weights: rag=0.80, specialist=0.15, dnn=0.05 | val_MAE=$31.57


{'w1_rag': 0.8,
 'w2_specialist': 0.15,
 'w3_dnn': 0.05,
 'mae': 31.56812305584717}

evaluate() now returns the Tester object, so results stay accessible for the rest of the notebook runtime.

Use it like this

`tester = evaluate(ensemble, test)
tester.worst_misses(20)
You can also inspect raw arrays directly if useful:
`

`tester.titles
tester.guesses
tester.truths
tester.errors`

In [34]:
tester = evaluate(ensemble, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$4 $40 $19 $11 $29 $134 $7 $25 $8 $56 $46 $57 $1 $6 $10 $1 $9 $24 $36 $9 $21 $13 $16 $26 $27 $196 $123 $0 $96 $53 $4 $4 $24 $1 $8 $82 $46 $34 $29 $12 $81 $36 $2 $37 $70 $3 $24 $1 $84 $3 $12 $52 $215 $7 $37 $29 $16 $43 $78 $8 $126 $37 $3 $60 $223 $11 $46 $278 $3 $12 $13 $8 $17 $4 $16 $13 $11 $3 $1 $15 $7 $10 $1 $46 $7 $2 $31 $36 $66 $1 $10 $5 $1 $5 $1 $18 $6 $45 $59 $95 $6 $1 $5 $2 $10 $45 $1 $270 $6 $127 $13 $6 $0 $42 $21 $2 $0 $9 $32 $92 $3 $38 $6 $0 $20 $46 $4 $20 $3 $24 $6 $42 $3 $2 $59 $1 $79 $14 $53 $14 $21 $125 $20 $7 $12 $25 $0 $14 $2 $5 $0 $6 $0 $3 $10 $0 $39 $2 $16 $2 $42 $13 $6 $1 $156 $2 $176 $19 $6 $2 $17 $1 $125 $3 $27 $11 $3 $26 $20 $12 $41 $12 $20 $21 $17 $20 $49 $6 $20 $3 $13 $6 $6 $34 $0 $11 $26 $5 $11 $5 

In [ ]:
from log_utils import log_run_summary, best_run_so_far, read_run_history

log_run_summary(
    tester,
    run="price_bucket_db",
    config={
        "use_buckets": True,
        "use_clamp": True,
        "clamp_padding": 0.2,
        "prompt_version": "important_v1",
        "weights": {
            "rag": RAG_WEIGHT,
            "specialist": SPECIALIST_WEIGHT,
            "dnn": DNN_WEIGHT,
        },
    },
)

best_run_so_far()

Logged run summary -> logs\run_history.jsonl | avg_error=$29.88 | run=price_bucket_db
Best so far: $29.69 on 2026-04-01T21:27:42 (run=price_bucket_db, predictor=ensemble)


{'timestamp': '2026-04-01T21:27:42',
 'date': '2026-04-01',
 'run': 'price_bucket_db',
 'predictor': 'ensemble',
 'size': 200,
 'avg_error': 29.6933,
 'config.use_buckets': True,
 'config.use_clamp': True,
 'config.clamp_padding': 0.2,
 'config.prompt_version': 'important_v1',
 'config.weights.rag': 0.8,
 'config.weights.specialist': 0.1,
 'config.weights.dnn': 0.1}

In [36]:
read_run_history().sort_values("avg_error").head(10)

,timestamp,date,run,predictor,size,avg_error,config.use_buckets,config.use_clamp,config.clamp_padding,config.prompt_version,config.weights.rag,config.weights.specialist,config.weights.dnn
1,2026-04-01T21:27:42,2026-04-01,price_bucket_db,ensemble,200,29.6933,True,True,0.2,important_v1,0.8,0.10,0.10
2,2026-04-01T21:31:57,2026-04-01,price_bucket_db,ensemble,200,29.8778,True,True,0.2,important_v1,0.7,0.25,0.05
0,2026-04-01T21:15:51,2026-04-01,price_bucket_db,ensemble,200,30.1412,True,True,0.2,important_v1,0.8,0.10,0.10


In [ ]:
tester.worst_misses(20)

In [ ]:
rows = list(zip(tester.titles, tester.truths, tester.guesses, tester.errors))

# worst misses
worst = sorted(rows, key=lambda x: abs(x[3]), reverse=True)[:20]

print(worst)

best = sorted(rows, key=lambda x: abs(x[3]), reverse=False)[:20]

print(best)

In [ ]:
worst3 = tester.worst_miss_indices(3)
worst3

In [ ]:
for i in tester.worst_miss_indices(3):
    item = test[i]
    print(f"\n{item.title}")
    print(
        f"pred=${tester.guesses[i]:,.2f} actual=${tester.truths[i]:,.2f} err=${tester.errors[i]:,.2f}"
    )
    documents, prices = find_similars(item)
    print([doc.split("\n")[0] for doc in documents])  # just titles

In [ ]:
def show_similars_for_worst(tester, test, k=3):
    for rank, i in enumerate(tester.worst_miss_indices(k), start=1):
        item = test[i]
        print(f"\n=== Worst #{rank} (test[{i}]) ===")
        print(f"{item.title}")
        print(
            f"Predicted ${tester.guesses[i]:,.2f} | "
            f"Actual ${tester.truths[i]:,.2f} | "
            f"Error ${tester.errors[i]:,.2f}"
        )

        documents, prices = find_similars(item)
        print("Similar items (title - price):")
        for doc, price in zip(documents, prices):
            similar_title = doc.split("\n")[0].replace("Title:", "").strip()
            print(f"- {similar_title} - ${price:,.2f}")

In [ ]:
show_similars_for_worst(tester, test, k=8)

---

Agents below (for price is right):

---

In [ ]:
from agents.ensemble_agent import EnsembleAgent

agent = EnsembleAgent(collection)

In [ ]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [ ]:
agent.price("Shure MV7+ professional podcaster microphone with usb-c and XLR outputs")

In [ ]:
from agents.neural_network_agent import NeuralNetworkAgent

agent = NeuralNetworkAgent()

In [ ]:
agent.price("Shure MV7+ professional podcaster microphone with usb-c and XLR outputs")

In [ ]:
from agents.frontier_agent import FrontierAgent

agent = FrontierAgent(collection)
agent.price(
    "Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio"
)

In [ ]:
def show_similars_for_best(tester, test, k=3):
    best_indices = sorted(range(len(tester.errors)), key=lambda i: tester.errors[i])[:k]

    for rank, i in enumerate(best_indices, start=1):
        item = test[i]
        print(f"\n=== Best #{rank} (test[{i}]) ===")
        print(f"{item.title}")
        print(
            f"Predicted ${tester.guesses[i]:,.2f} | "
            f"Actual ${tester.truths[i]:,.2f} | "
            f"Error ${tester.errors[i]:,.2f}"
        )

        documents, prices = find_similars(item)
        print("Similar items (title - price):")
        for doc, price in zip(documents, prices):
            similar_title = doc.split("\n")[0].replace("Title:", "").strip()
            print(f"- {similar_title} - ${price:,.2f}")

In [ ]:
show_similars_for_best(tester, test, k=3)